In [ ]:
from pytoast.ocean.adcp import ADCP
import glob
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Files from test directory
files = glob.glob("../py-tests/src/ocean/adcp/testdata/BBASIT*.mat")
files.sort()

In [ ]:
name_map = {
    "u1": "Burst_VelBeam1",
    "u2": "Burst_VelBeam2",
    "u3": "Burst_VelBeam3",
    "u4": "Burst_VelBeam4",
    "u5": "IBurst_VelBeam5",
    "p": "Burst_Pressure",
    "time": "Burst_TimeStamp",
    "z": "Burst_Range",
    "heading": "Burst_Heading",
    "pitch": "Burst_Pitch",
    "roll": "Burst_Roll",
}

adcp = ADCP(
    files=files, name_map=name_map, source_coords="beam", orientation="up", manufacturer="nortek", data_keys="Data"
)

In [ ]:
# Tranformation matrix for the specific instrument
T = np.array([[1.1831, 0, 0.5518, 0], [0, -1.1831, 0, 0.5518], [-1.1831, 0, 0.5518, 0], [0, 1.1831, 0, 0.5518]])

# Preprocessing opts to despike and rotate to ENU
pre_opts = {
    "despike": {"method": "goring_nikora"},
    "rotate": {
        "flow_rotation": "align_streamwise",
        "coords_out": "xyz",
        "transformation_matrix": T,
    },
}
adcp.set_preprocess_opts(pre_opts)

In [ ]:
# Looping through the bursts to compare reynolds stress and dissipation estimates
cov_varmethod = np.empty((adcp.n_bursts, adcp.n_heights))
cov_ogive = np.empty((adcp.n_bursts, adcp.n_heights))

for ii in range(adcp.n_bursts):
    burst = adcp.load_burst(ii)
    cov_varmethod[ii, :] = adcp.covariance(burst, method="variance")["uw"]
    cov_ogive[ii, :] = adcp.covariance(burst, method="ogive_fit", ogive_r2_min=0.7)["uw"]

In [ ]:
# Comparison plot
plot_idx = 2
fig, ax1 = plt.subplots(figsize=(5, 4))
ax1.plot(cov_varmethod[plot_idx, :], adcp.z, "o-", label="Variance")
ax1.plot(cov_ogive[plot_idx, :], adcp.z, 'o-', alpha=0.5, label="Ogive fit")
ax1.set_xlabel(r"Reynolds stress (m$^2$/s$^2$)")
ax1.set_ylabel(r"Distance above bed (m)")
ax1.legend()
fig.tight_layout(pad=0.5)